# Phase D / Experiment 1 — Chunk Size × Overlap (RQ1)

This notebook implements **MASTER_PLAN.md §10 D.1** (Exp1: chunk-size × overlap
sweep) under the conventions in §13.5 (determinism) and §15.

**Chained-batches design.** The notebook is parametrised by a top-of-file
tuple `BATCH_CHUNK_SIZES_TO_RUN`. A single **Run All** sweeps every chunk size
in the tuple back-to-back; cell 6 wraps the per-batch flow in a top-level
`for BATCH_CHUNK_SIZE in BATCH_CHUNK_SIZES_TO_RUN` loop. Cells 3 (one-time
setup) and 5 (helper definition) execute once; cell 7 prints a combined
summary grouped by batch `chunk_size`.

| Run | Tuple | Configs swept (overlap_pct ∈ {0, 10, 20}) |
|----:|-------|-------------------------------------------|
|  1  | `(256,)`           | 3 configs — already done on 2026-05-10 |
|  2  | `(512, 1024)`      | 6 configs — chained, one Run All |

Per batch (one chunk size): 3 indexing rows + 117 query rows appended to
`results/indexing_log.csv` / `results/query_log.csv`. Across the full Exp1:
**9 indexing rows + 351 query rows**.

**Repetition policy (§10 D.4).** For each config, the index is built **once**
(deterministic under fixed seed 42) and the 13-question query loop is run
**three times**. Per-config row count: 1 indexing + 39 query = 40 rows.

**Run-id scheme (Phase D pre-locked decision).**
- indexing run_id = `{ts_compact}_{config_hash}`
- query run_id, rep r ∈ {1, 2, 3} = `{indexing_run_id}_r{r}`

So the indexing run_id is a strict prefix of the matching query run_ids;
Phase E joins are `df.run_id.str.startswith(idx_run_id)`. Each config gets
its own UTC anchor inside `run_one_config` (cell 5), so chaining batches in
one Run All keeps every `idx_run_id` unique.

**CodeCarbon (§13.4).** `OfflineEmissionsTracker` is used everywhere
(CodeCarbon 3.2.6 dropped `country_iso_code` from the online tracker —
recorded in the Phase C entry of PROGRESS_LOG.md). One indexing tracker
per config; one query tracker per (config, repetition) block. Per-query
energy/CO2 is proportionally allocated from the block tracker by
`total_latency_ms` weight (Phase C decision Q2).


In [7]:
"""Cell 2 — Loop driver. A single Run All sweeps every chunk size in this
tuple back-to-back via the top-level for-loop in cell 6.

Per-batch row count appended to results/{indexing,query}_log.csv:
3 indexing rows + 117 query rows (3 overlap_pcts × 3 reps × 13 questions).

Batch 1 (chunk_size=256) was completed on 2026-05-10 in 26.2 min wall.
Do NOT include 256 in this tuple unless you want to re-run it — that would
generate fresh run_ids under the same config_hashes and produce duplicate
measurements in the cumulative CSVs. The default below picks up the
remaining (512, 1024) chained back-to-back.
"""

BATCH_CHUNK_SIZES_TO_RUN: tuple[int, ...] = (512, 1024)


In [8]:
"""Cell 3 — Imports, paths, determinism guards, eval questions.

One-time setup. Runs once regardless of how many chunk sizes are in
BATCH_CHUNK_SIZES_TO_RUN.
"""
from __future__ import annotations

import random
import sys
import time
import tracemalloc
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Locate project root (handles both `cwd=project_root` and `cwd=notebooks/`).
_HERE = Path.cwd()
ROOT = _HERE if (_HERE / "MASTER_PLAN.md").exists() else _HERE.parent
assert (ROOT / "MASTER_PLAN.md").exists(), f"Could not find project root from {_HERE}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.chunking import chunk_corpus
from src.embedding import embed_chunks
from src.eval_questions import load_eval_questions
from src.indexing import build_index, save_index
from src.metrics import (
    EXPECTED_INDEXING_COLS,
    EXPECTED_QUERY_COLS,
    NOTES_TAG_PROP_ENERGY,
    PeakRAMSampler,
    allocate_block_energy_proportionally,
    assert_indexing_schema,
    assert_query_sanity,
    assert_query_schema,
    count_retrieved_tokens,
    count_text_tokens,
    make_run_id,
    set_global_seeds,
    timing_context,
)
from src.pipeline import RAGConfig, run_rag

# §9 C.5 — determinism guards (notebook-side, in addition to Ollama defaults).
random.seed(42)
np.random.seed(42)
set_global_seeds(42)

# Validate the driver tuple from cell 2 — list-level checks now that we
# accept >1 batch per Run All.
ALLOWED_BATCH_CHUNK_SIZES = {256, 512, 1024}
assert isinstance(BATCH_CHUNK_SIZES_TO_RUN, tuple), (
    "BATCH_CHUNK_SIZES_TO_RUN must be a tuple of ints (cell 2)"
)
assert len(BATCH_CHUNK_SIZES_TO_RUN) >= 1, (
    "BATCH_CHUNK_SIZES_TO_RUN is empty — nothing to run"
)
for _b in BATCH_CHUNK_SIZES_TO_RUN:
    assert _b in ALLOWED_BATCH_CHUNK_SIZES, (
        f"BATCH_CHUNK_SIZES_TO_RUN contains {_b!r}; allowed values are "
        f"{sorted(ALLOWED_BATCH_CHUNK_SIZES)}"
    )
assert len(set(BATCH_CHUNK_SIZES_TO_RUN)) == len(BATCH_CHUNK_SIZES_TO_RUN), (
    f"duplicate values in BATCH_CHUNK_SIZES_TO_RUN={BATCH_CHUNK_SIZES_TO_RUN}"
)

# Project paths.
RESULTS_DIR  = ROOT / "results"
CARBON_DIR   = RESULTS_DIR / "carbon"
INDICES_DIR  = ROOT / "indices"
CORPUS_TXT   = ROOT / "corpus_text"
EVAL_QS_PATH = ROOT / "dataset" / "eval_questions.md"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CARBON_DIR.mkdir(parents=True, exist_ok=True)
INDICES_DIR.mkdir(parents=True, exist_ok=True)

# Load the 13 eval questions — same parser used by Phase B/C.
eval_questions = load_eval_questions(EVAL_QS_PATH)
type_counts: dict[str, int] = {}
for q in eval_questions:
    type_counts[q.question_type] = type_counts.get(q.question_type, 0) + 1

print(f"[D.1] Project root: {ROOT}")
print(f"[D.1] BATCH_CHUNK_SIZES_TO_RUN = {BATCH_CHUNK_SIZES_TO_RUN}")
print(f"[D.1] Loaded {len(eval_questions)} eval questions; by type: {type_counts}")


[D.1] Project root: <project_root>
[D.1] BATCH_CHUNK_SIZES_TO_RUN = (512, 1024)
[D.1] Loaded 13 eval questions; by type: {'factoid': 6, 'synthesis': 3, 'numerical': 3, 'multi-doc': 1}


In [9]:
"""Cell 4 — Pre-flight: list every config that will be swept this Run All.

Catches duplicate config_hashes across batches before any indexing work
happens. Configs are RE-built fresh inside the cell-6 loop so the
overlap_pct list and config_hash logic stay co-located with the sweep.
"""

OVERLAP_PCTS_TO_SWEEP = (0, 10, 20)  # §10 D.1 grid (one axis)

_planned_configs = [
    RAGConfig(chunk_size=cs, overlap_pct=op)
    for cs in BATCH_CHUNK_SIZES_TO_RUN
    for op in OVERLAP_PCTS_TO_SWEEP
]
_planned_hashes = [c.hash for c in _planned_configs]
assert len(set(_planned_hashes)) == len(_planned_hashes), (
    f"BUG: duplicate config_hash among planned configs: "
    f"{list(zip(_planned_hashes, _planned_configs))}"
)

print(f"[D.1] Pre-flight: {len(_planned_configs)} configs across "
      f"{len(BATCH_CHUNK_SIZES_TO_RUN)} batch(es)")
for cs in BATCH_CHUNK_SIZES_TO_RUN:
    print(f"  batch chunk_size={cs}:")
    for op in OVERLAP_PCTS_TO_SWEEP:
        cfg = RAGConfig(chunk_size=cs, overlap_pct=op)
        print(f"    hash={cfg.hash}  chunk_size={cfg.chunk_size}  "
              f"overlap_pct={op:>2}%  embedding={cfg.embedding_model}")


[D.1] Pre-flight: 6 configs across 2 batch(es)
  batch chunk_size=512:
    hash=a9e9b27f  chunk_size=512  overlap_pct= 0%  embedding=sentence-transformers/all-MiniLM-L6-v2
    hash=6fb92835  chunk_size=512  overlap_pct=10%  embedding=sentence-transformers/all-MiniLM-L6-v2
    hash=aadb0cb9  chunk_size=512  overlap_pct=20%  embedding=sentence-transformers/all-MiniLM-L6-v2
  batch chunk_size=1024:
    hash=7dffb047  chunk_size=1024  overlap_pct= 0%  embedding=sentence-transformers/all-MiniLM-L6-v2
    hash=1fbb3d49  chunk_size=1024  overlap_pct=10%  embedding=sentence-transformers/all-MiniLM-L6-v2
    hash=b1843394  chunk_size=1024  overlap_pct=20%  embedding=sentence-transformers/all-MiniLM-L6-v2


In [10]:
"""Cell 5 — Helper: run one configuration end-to-end (1 index + 3 query reps).

UNCHANGED from the batch-1 run (which already produced 3 indexing + 117 query
rows for chunk_size=256 with all sanity asserts passing). Each call:

  1. Anchors a UTC timestamp once. This anchor is shared by the indexing
     run_id and all 3 query run_ids — guarantees that idx_run_id is a strict
     prefix of every query run_id from this config (Phase D run-id scheme).
  2. Builds the index ONCE under an OfflineEmissionsTracker, PeakRAMSampler,
     timing_context, and tracemalloc (Phase C C.1–C.4 instrumentation reused).
     Appends 1 row to results/indexing_log.csv with run_id = idx_run_id.
  3. For r ∈ {1, 2, 3}: opens a fresh OfflineEmissionsTracker, runs all 13
     questions, allocates block energy/CO2 proportionally by total_latency_ms,
     appends 13 rows with run_id = `${idx_run_id}_r${r}`.
  4. After all 3 reps: re-reads both CSVs, runs schema lock-in + sanity
     asserts (A–H from src.metrics.assert_query_sanity) on rows filtered to
     this config's run_id (prefix match).
"""
from codecarbon import OfflineEmissionsTracker

REFUSAL_PHRASE = "I don't have enough information"


def run_one_config(cfg: RAGConfig) -> dict:
    # 1) One UTC anchor for the whole config (indexing + 3 reps).
    config_ts = datetime.now(timezone.utc)
    idx_ts_iso, idx_ts_compact, idx_run_id = make_run_id(
        cfg.hash, ts=config_ts,
    )

    print()
    print("=" * 72)
    print(f"[D.1] Config: chunk_size={cfg.chunk_size}, "
          f"overlap_pct={cfg.overlap_pct}%, hash={cfg.hash}")
    print(f"[D.1] idx_run_id = {idx_run_id}")
    print("=" * 72)

    # 2) Indexing block — built ONCE per config.
    INDEX_DIR = INDICES_DIR / cfg.hash

    idx_tracker = OfflineEmissionsTracker(
        project_name=f"phase_d_exp1_indexing_{cfg.hash}",
        tracking_mode="machine",
        country_iso_code="DEU",
        output_dir=str(CARBON_DIR),
        output_file="emissions.csv",
        measure_power_secs=1.0,
        log_level="error",
        save_to_file=True,
    )

    tracemalloc.start()
    idx_tracker.start()
    try:
        with timing_context() as IDX_T, PeakRAMSampler(interval_sec=0.1) as RAM:
            chunks = chunk_corpus(
                text_dir=CORPUS_TXT,
                chunk_size=cfg.chunk_size,
                overlap_pct=cfg.overlap_pct,
            )
            embeddings = embed_chunks(
                chunks,
                model_name=cfg.embedding_model,
                show_progress=False,
            )
            faiss_index = build_index(embeddings)
            save_index(faiss_index, chunks, INDEX_DIR)
    finally:
        idx_co2_kg = idx_tracker.stop()
        _, tm_peak_b = tracemalloc.get_traced_memory()
        tracemalloc.stop()

    idx_energy_wh = idx_tracker.final_emissions_data.energy_consumed * 1000.0
    idx_co2_g     = idx_co2_kg * 1000.0
    INDEX_SIZE_MB = (INDEX_DIR / "faiss.index").stat().st_size / 1_000_000.0

    INDEXING_TIME_SEC = (
        IDX_T.get("chunking", 0.0)
        + IDX_T.get("embedding", 0.0)
        + IDX_T.get("faiss_build", 0.0)
        + IDX_T.get("index_save", 0.0)
    )

    n_documents = len({c.source_doc for c in chunks})

    indexing_row = {
        "run_id":               idx_run_id,
        "timestamp":            idx_ts_iso,
        "config_hash":          cfg.hash,
        "chunk_size":           int(cfg.chunk_size),
        "chunk_overlap":        int(cfg.overlap_pct),  # Q3: percentage
        "embedding_model":      cfg.embedding_model,
        "n_documents":          int(n_documents),
        "n_chunks_total":       int(len(chunks)),
        "indexing_time_sec":    float(INDEXING_TIME_SEC),
        "peak_ram_mb":          float(RAM.peak_rss_mb),
        "index_size_mb":        float(INDEX_SIZE_MB),
        "embedding_time_sec":   float(IDX_T.get("embedding", 0.0)),
        "faiss_build_time_sec": float(IDX_T.get("faiss_build", 0.0)),
        "energy_wh":            float(idx_energy_wh),
        "co2_g":                float(idx_co2_g),
        "notes":                "phase_d_exp1",
    }
    idx_csv = RESULTS_DIR / "indexing_log.csv"
    write_header = not idx_csv.exists()
    pd.DataFrame([indexing_row], columns=EXPECTED_INDEXING_COLS).to_csv(
        idx_csv, mode="a", index=False, header=write_header,
    )

    print(f"[D.1] Indexed: n_chunks={len(chunks)}, "
          f"indexing_time={INDEXING_TIME_SEC:.2f}s, "
          f"peak_ram={RAM.peak_rss_mb:.1f} MB, "
          f"index_size={INDEX_SIZE_MB:.2f} MB, "
          f"energy={idx_energy_wh:.4f} Wh, "
          f"tracemalloc_peak={tm_peak_b/1_000_000:.2f} MB (Q4: not in CSV)")

    # 3) Three query reps. Index/chunks reused across reps.
    rep_summaries: list[dict] = []
    for r in (1, 2, 3):
        rep_ts_iso, _, qry_run_id = make_run_id(
            cfg.hash, ts=config_ts, repetition=r,
        )
        assert qry_run_id.startswith(idx_run_id), (
            f"BUG: qry_run_id={qry_run_id!r} not prefixed by "
            f"idx_run_id={idx_run_id!r}"
        )

        qry_tracker = OfflineEmissionsTracker(
            project_name=f"phase_d_exp1_query_{cfg.hash}_r{r}",
            tracking_mode="machine",
            country_iso_code="DEU",
            output_dir=str(CARBON_DIR),
            output_file="emissions.csv",
            measure_power_secs=1.0,
            log_level="error",
            save_to_file=True,
        )

        rows: list[dict] = []
        rep_t0 = time.perf_counter()
        qry_tracker.start()
        try:
            for q in eval_questions:
                result = run_rag(q.question, cfg, faiss_index, chunks)
                t = result["timings"]
                retrieved = result["retrieved_chunks"]

                embed_ms    = t["embed"]    * 1000.0
                retrieve_ms = t["retrieve"] * 1000.0
                generate_ms = t["generate"] * 1000.0
                total_ms    = embed_ms + retrieve_ms + generate_ms

                rows.append({
                    "run_id":                qry_run_id,
                    "timestamp":             rep_ts_iso,
                    "config_hash":           cfg.hash,
                    "question_id":           q.question_id,
                    "top_k":                 int(cfg.top_k),
                    "query_embed_time_ms":   float(embed_ms),
                    "retrieval_time_ms":     float(retrieve_ms),
                    "generation_time_ms":    float(generate_ms),
                    "total_latency_ms":      float(total_ms),
                    "n_retrieved_chunks":    int(len(retrieved)),
                    "retrieved_token_count": int(count_retrieved_tokens(retrieved)),
                    "prompt_token_count":    int(count_text_tokens(result["prompt"])),
                    "answer_token_count":    int(count_text_tokens(result["answer"])),
                    "energy_wh_per_query":   None,
                    "co2_g_per_query":       None,
                    "retrieved_chunk_ids":   ";".join(rr.chunk.chunk_id for rr in retrieved),
                    "answer_text":           result["answer"],
                    "notes":                 "",
                })
        finally:
            qry_co2_kg = qry_tracker.stop()
        rep_wall_s = time.perf_counter() - rep_t0

        qry_energy_wh = qry_tracker.final_emissions_data.energy_consumed * 1000.0
        qry_co2_g     = qry_co2_kg * 1000.0

        allocate_block_energy_proportionally(
            rows, qry_energy_wh, qry_co2_g,
        )

        qry_csv = RESULTS_DIR / "query_log.csv"
        write_header = not qry_csv.exists()
        pd.DataFrame(rows, columns=EXPECTED_QUERY_COLS).to_csv(
            qry_csv, mode="a", index=False, header=write_header,
        )

        n_refusals = sum(
            1 for row in rows if REFUSAL_PHRASE in row["answer_text"]
        )
        mean_total_ms = sum(row["total_latency_ms"] for row in rows) / len(rows)
        rep_summaries.append({
            "rep": r,
            "qry_run_id": qry_run_id,
            "energy_wh": qry_energy_wh,
            "co2_g": qry_co2_g,
            "n_refusals": n_refusals,
            "mean_total_ms": mean_total_ms,
            "rep_wall_s": rep_wall_s,
        })
        print(f"[D.1]   rep {r}: wall={rep_wall_s:.1f}s "
              f"mean_total_latency={mean_total_ms/1000:.1f}s "
              f"refusals={n_refusals}/13 "
              f"qry_energy={qry_energy_wh:.4f} Wh  "
              f"qry_run_id={qry_run_id}")

    # 4) Sanity asserts AFTER all 3 reps for this config.
    idx_df_full = pd.read_csv(RESULTS_DIR / "indexing_log.csv")
    qry_df_full = pd.read_csv(RESULTS_DIR / "query_log.csv")

    assert_indexing_schema(idx_df_full)
    assert_query_schema(qry_df_full)

    idx_this = idx_df_full[idx_df_full["run_id"] == idx_run_id]
    qry_this = qry_df_full[
        qry_df_full["run_id"].astype(str).str.startswith(idx_run_id)
    ]

    assert_query_sanity(
        qry_this, idx_this,
        top_k=cfg.top_k,
        chunk_size=cfg.chunk_size,
        expected_query_count=39,
        expected_indexing_count=1,
        notes_tag=NOTES_TAG_PROP_ENERGY,
    )
    print(f"[D.1] Sanity asserts (schema lock + A–H) PASSED for "
          f"hash={cfg.hash} (39 query rows, 1 indexing row)")

    return {
        "idx_run_id":         idx_run_id,
        "config_hash":        cfg.hash,
        "chunk_size":         cfg.chunk_size,
        "overlap_pct":        cfg.overlap_pct,
        "n_chunks_total":     len(chunks),
        "indexing_time_sec":  INDEXING_TIME_SEC,
        "peak_ram_mb":        RAM.peak_rss_mb,
        "index_size_mb":      INDEX_SIZE_MB,
        "idx_energy_wh":      idx_energy_wh,
        "idx_co2_g":          idx_co2_g,
        "rep_summaries":      rep_summaries,
    }


In [11]:
"""Cell 6 — Top-level loop. Sweeps every chunk size in the driver tuple.

   for BATCH_CHUNK_SIZE in BATCH_CHUNK_SIZES_TO_RUN:
       <existing per-batch flow>

Each iteration builds 3 RAGConfigs (one per overlap_pct), runs them through
run_one_config(), and accumulates the per-config summaries. Total wall time
is reported at the end.
"""

all_batch_results: list[dict] = []
total_t0 = time.perf_counter()

for BATCH_CHUNK_SIZE in BATCH_CHUNK_SIZES_TO_RUN:
    print()
    print("#" * 72)
    print(f"# BEGIN BATCH  chunk_size={BATCH_CHUNK_SIZE}")
    print("#" * 72)

    configs_to_sweep: list[RAGConfig] = [
        RAGConfig(chunk_size=BATCH_CHUNK_SIZE, overlap_pct=op)
        for op in OVERLAP_PCTS_TO_SWEEP
    ]
    # Defensive: hashes within a batch must be unique (different overlap_pct
    # → different hash). The cell-4 pre-flight already checks across batches.
    _hashes = [c.hash for c in configs_to_sweep]
    assert len(set(_hashes)) == len(_hashes), (
        f"BUG: duplicate config_hash within batch chunk_size={BATCH_CHUNK_SIZE}: "
        f"{_hashes}"
    )

    batch_t0 = time.perf_counter()
    for cfg in configs_to_sweep:
        summary = run_one_config(cfg)
        summary["batch_chunk_size"] = BATCH_CHUNK_SIZE  # for cell-7 grouping
        all_batch_results.append(summary)
    batch_wall_s = time.perf_counter() - batch_t0

    print()
    print(f"[D.1] END BATCH  chunk_size={BATCH_CHUNK_SIZE}: "
          f"wall={batch_wall_s/60:.1f} min ({batch_wall_s:.0f} s) "
          f"for {len(configs_to_sweep)} configs.")

total_wall_s = time.perf_counter() - total_t0
print()
print(f"[D.1] Total wall across {len(BATCH_CHUNK_SIZES_TO_RUN)} batch(es): "
      f"{total_wall_s/60:.1f} min ({total_wall_s:.0f} s).")



########################################################################
# BEGIN BATCH  chunk_size=512
########################################################################

[D.1] Config: chunk_size=512, overlap_pct=0%, hash=a9e9b27f
[D.1] idx_run_id = 20260510T070513Z_a9e9b27f
[D.1] Indexed: n_chunks=714, indexing_time=4.50s, peak_ram=291.6 MB, index_size=1.10 MB, energy=0.0569 Wh, tracemalloc_peak=7.37 MB (Q4: not in CSV)
[D.1]   rep 1: wall=206.5s mean_total_latency=15.9s refusals=3/13 qry_energy=2.6088 Wh  qry_run_id=20260510T070513Z_a9e9b27f_r1
[D.1]   rep 2: wall=237.0s mean_total_latency=18.2s refusals=3/13 qry_energy=2.9954 Wh  qry_run_id=20260510T070513Z_a9e9b27f_r2
[D.1]   rep 3: wall=246.5s mean_total_latency=19.0s refusals=3/13 qry_energy=3.1147 Wh  qry_run_id=20260510T070513Z_a9e9b27f_r3
[D.1] Sanity asserts (schema lock + A–H) PASSED for hash=a9e9b27f (39 query rows, 1 indexing row)

[D.1] Config: chunk_size=512, overlap_pct=10%, hash=6fb92835
[D.1] idx_run_id = 20260

In [12]:
"""Cell 7 — Combined summary across all batches in this Run All."""

print()
print("=" * 72)
print(f"Phase D / Exp1 — Combined summary across "
      f"{len(BATCH_CHUNK_SIZES_TO_RUN)} batch(es) "
      f"({BATCH_CHUNK_SIZES_TO_RUN})")
print("=" * 72)

for cs in BATCH_CHUNK_SIZES_TO_RUN:
    print()
    print(f"--- batch chunk_size={cs} ---")
    batch_subset = [s for s in all_batch_results if s["batch_chunk_size"] == cs]
    for s in batch_subset:
        n_ref_per_rep    = [rep["n_refusals"]    for rep in s["rep_summaries"]]
        mean_lat_per_rep = [rep["mean_total_ms"] for rep in s["rep_summaries"]]
        energy_per_rep   = [rep["energy_wh"]     for rep in s["rep_summaries"]]
        mean_refusal_rate = sum(n_ref_per_rep) / (3 * 13)
        print()
        print(f"  config: chunk_size={s['chunk_size']:>4} "
              f"overlap={s['overlap_pct']:>2}%  hash={s['config_hash']}")
        print(f"    n_chunks_total={s['n_chunks_total']:>4}  "
              f"indexing_time={s['indexing_time_sec']:.2f}s  "
              f"peak_ram_mb={s['peak_ram_mb']:.1f}  "
              f"index_size_mb={s['index_size_mb']:.2f}")
        print(f"    idx_energy_wh={s['idx_energy_wh']:.4f}  "
              f"idx_co2_g={s['idx_co2_g']:.4f}")
        print(f"    refusals/rep   : {n_ref_per_rep}  "
              f"-> mean_refusal_rate={mean_refusal_rate:.1%}")
        print(f"    mean_total_ms/rep: {[f'{x:.0f}' for x in mean_lat_per_rep]}")
        print(f"    qry_energy_wh/rep: {[f'{x:.4f}' for x in energy_per_rep]}")

print()
print(f"[D.1] Wrote {len(all_batch_results)} indexing rows + "
      f"{len(all_batch_results) * 39} query rows total in this Run All.")



Phase D / Exp1 — Combined summary across 2 batch(es) ((512, 1024))

--- batch chunk_size=512 ---

  config: chunk_size= 512 overlap= 0%  hash=a9e9b27f
    n_chunks_total= 714  indexing_time=4.50s  peak_ram_mb=291.6  index_size_mb=1.10
    idx_energy_wh=0.0569  idx_co2_g=0.0217
    refusals/rep   : [3, 3, 3]  -> mean_refusal_rate=23.1%
    mean_total_ms/rep: ['15877', '18230', '18956']
    qry_energy_wh/rep: ['2.6088', '2.9954', '3.1147']

  config: chunk_size= 512 overlap=10%  hash=6fb92835
    n_chunks_total= 754  indexing_time=5.39s  peak_ram_mb=289.2  index_size_mb=1.16
    idx_energy_wh=0.0681  idx_co2_g=0.0259
    refusals/rep   : [4, 4, 4]  -> mean_refusal_rate=30.8%
    mean_total_ms/rep: ['19194', '19465', '18037']
    qry_energy_wh/rep: ['3.1538', '3.1983', '2.9637']

  config: chunk_size= 512 overlap=20%  hash=aadb0cb9
    n_chunks_total= 821  indexing_time=5.72s  peak_ram_mb=337.0  index_size_mb=1.26
    idx_energy_wh=0.0723  idx_co2_g=0.0276
    refusals/rep   : [2, 2, 2] 

## Wrap-up — what this notebook produced

For each chunk size in `BATCH_CHUNK_SIZES_TO_RUN`, three configs are swept
(overlap_pct ∈ {0, 10, 20}); for each config the index is built once and the
13-question evaluation is run three times. Every config produces:

- **1 row** appended to `results/indexing_log.csv`.
- **39 rows** appended to `results/query_log.csv` (3 reps × 13 questions).
- **Two emission blocks per rep** in `results/carbon/emissions.csv`
  (`project_name=phase_d_exp1_indexing_<hash>` and
   `project_name=phase_d_exp1_query_<hash>_r{r}`).
- **Sanity asserts run after each config's rows are appended** — schema lock-in
  on both CSVs, plus A–H value asserts (`src.metrics.assert_query_sanity`)
  on rows filtered by `run_id.str.startswith(idx_run_id)`.

A single Run All sweeps every chunk size in the driver tuple back-to-back; the
top-level for-loop in cell 6 wraps the per-batch flow. Every config gets a
fresh UTC anchor inside `run_one_config`, so chaining batches keeps every
`idx_run_id` unique.

**To select the Exp1 winner** (after all three batches are done — i.e. all of
{256, 512, 1024} have appeared in some Run All across the project history),
use the pre-locked lexicographic Pareto rule from the prior task:

1. Primary: lowest mean refusal rate over the 3 reps (refusal := answer
   contains the literal phrase `"I don't have enough information"`).
2. Tie-break #1: lowest mean `total_latency_ms` over (3 reps × 13 questions).
3. Tie-break #2: lowest `indexing_time_sec`.

The winner's `(chunk_size, overlap_pct)` is then frozen as the Exp2 baseline
in `notebooks/04_exp2_embedding_model.ipynb`.
